# 86 — Grad-CAM Error Analysis (3-class Primer conf60)

**Tujuan:** Analisis *mengapa* model salah prediksi — region wajah mana yang diaktivasi model saat membuat keputusan yang benar vs salah.

**4 model yang dibandingkan (satu per strategi fusion):**
| Model | Fusion Strategy | CNN Branch |
|:---|:---|:---|
| `CNN_TL` | None (image only) | ResNet18 |
| `EarlyFusion_TL` | Early (input-level) | ResNet18 4-channel |
| `Intermediate_TL` | Intermediate (feature-level) | ResNet18 |
| `LateFusion_TL` | Late (decision-level) | ResNet18 (CNN branch) |

**Grad-CAM target layer:** `layer4` terakhir dari ResNet18 backbone (`features[-2][-1]`)

**Analisis yang dilakukan:**
1. Identifikasi sampel salah prediksi (misclassified) per kelas
2. Grad-CAM overlay untuk salah prediksi vs benar prediksi
3. Perbandingan antar 4 fusion strategy — apakah landmark mengubah fokus region?

**Prerequisite:** Jalankan `scripts/retrain_for_gradcam.py` terlebih dahulu untuk menyimpan checkpoint.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from sklearn.metrics import f1_score, confusion_matrix, classification_report

from training.models import (
    EmotionCNNTransfer,
    EmotionEarlyFusionTransfer,
    EmotionFCNN,
    IntermediateFusionTransfer,
)

DATA_DIR  = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
CKPT_DIR  = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'gradcam_ckpts'
OUT_DIR   = PROJECT_ROOT / 'outputs' / 'gradcam'
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 3
REMAP_3 = np.array([1, 0, 2, 2, 2, 2, 0], dtype=np.int64)
CLASS_NAMES = ['positive', 'neutral', 'negative']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# ── Load test data ──────────────────────────────────────────────────────────
img_te = np.load(DATA_DIR / 'X_test_images.npy').astype(np.float32)
lm_te  = np.load(DATA_DIR / 'X_test_landmarks.npy').astype(np.float32)
hm_te  = np.load(DATA_DIR / 'X_test_heatmaps.npy').astype(np.float32)
y7_te  = np.load(DATA_DIR / 'y_test.npy')
y_te   = REMAP_3[y7_te].astype(np.int64)

print(f'Test samples: {len(y_te)}')
print(f'Class dist: {np.bincount(y_te, minlength=NUM_CLASSES).tolist()}')
print(f'  → {dict(zip(CLASS_NAMES, np.bincount(y_te, minlength=NUM_CLASSES).tolist()))}')

In [ ]:
# ── Helper: prepare tensor per model type ──────────────────────────────────
def img_tensor(idx):
    """(N,H,W,3) → (N,3,H,W) float tensor"""
    return torch.from_numpy(img_te[idx]).permute(0, 3, 1, 2).float().to(DEVICE)

def ef_tensor(idx):
    """EarlyFusion: (N,4,H,W) = RGB + heatmap channel"""
    t_img = torch.from_numpy(img_te[idx]).permute(0, 3, 1, 2).float()
    t_hm  = torch.from_numpy(hm_te[idx]).unsqueeze(1).float()
    return torch.cat([t_img, t_hm], dim=1).to(DEVICE)

def lm_tensor(idx):
    return torch.from_numpy(lm_te[idx]).float().to(DEVICE)


# ── Helper: get predictions for all test samples ────────────────────────────
def get_predictions(model, model_type, batch_size=64):
    model.eval()
    all_preds, all_probs = [], []
    n = len(y_te)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            idx = list(range(i, min(i + batch_size, n)))
            if model_type == 'cnn':
                out = model(img_tensor(idx))
            elif model_type == 'early_fusion':
                out = model(ef_tensor(idx))
            elif model_type == 'fusion':
                out = model(img_tensor(idx), lm_tensor(idx))
            probs = torch.softmax(out, dim=1).cpu().numpy()
            preds = probs.argmax(1)
            all_preds.append(preds)
            all_probs.append(probs)
    return np.concatenate(all_preds), np.concatenate(all_probs)

print('Helpers defined.')

In [ ]:
# ── Load 4 models ───────────────────────────────────────────────────────────
def load_ckpt(model, path):
    model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    model.eval()
    return model

models = {}
model_types = {}

# 1. CNN_TL
ckpt = CKPT_DIR / 'cnn_tl.pth'
assert ckpt.exists(), f'Missing: {ckpt}. Run scripts/retrain_for_gradcam.py first'
models['CNN_TL'] = load_ckpt(EmotionCNNTransfer(num_classes=NUM_CLASSES).to(DEVICE), ckpt)
model_types['CNN_TL'] = 'cnn'
print('CNN_TL loaded')

# 2. EarlyFusion_TL
ckpt = CKPT_DIR / 'early_fusion_tl.pth'
assert ckpt.exists(), f'Missing: {ckpt}'
models['EarlyFusion_TL'] = load_ckpt(EmotionEarlyFusionTransfer(num_classes=NUM_CLASSES).to(DEVICE), ckpt)
model_types['EarlyFusion_TL'] = 'early_fusion'
print('EarlyFusion_TL loaded')

# 3. Intermediate_TL
ckpt = CKPT_DIR / 'intermediate_tl.pth'
assert ckpt.exists(), f'Missing: {ckpt}'
models['Intermediate_TL'] = load_ckpt(IntermediateFusionTransfer(num_classes=NUM_CLASSES).to(DEVICE), ckpt)
model_types['Intermediate_TL'] = 'fusion'
print('Intermediate_TL loaded')

# 4. LateFusion_TL — CNN branch only for Grad-CAM
ckpt_cnn = CKPT_DIR / 'late_fusion_tl_cnn.pth'
assert ckpt_cnn.exists(), f'Missing: {ckpt_cnn}'
models['LateFusion_TL'] = load_ckpt(EmotionCNNTransfer(num_classes=NUM_CLASSES).to(DEVICE), ckpt_cnn)
model_types['LateFusion_TL'] = 'cnn'
print('LateFusion_TL (CNN branch) loaded')

In [ ]:
# ── Get predictions + metrics per model ─────────────────────────────────────
all_preds = {}
all_probs = {}

for name, model in models.items():
    preds, probs = get_predictions(model, model_types[name])
    all_preds[name] = preds
    all_probs[name] = probs
    f1 = f1_score(y_te, preds, average='macro', zero_division=0)
    print(f'{name:20s}: macro F1 = {f1:.4f}')

# LateFusion_TL: ensemble CNN + FCNN
import json
retrain_results = json.load(open(CKPT_DIR / 'retrain_results.json'))
best_w = retrain_results.get('late_fusion_tl', {}).get('best_w_cnn', 0.5)
print(f'\nLateFusion_TL best_w_cnn = {best_w:.2f} (loaded from retrain results)')

fcnn_model = load_ckpt(EmotionFCNN(num_classes=NUM_CLASSES).to(DEVICE),
                       CKPT_DIR / 'late_fusion_tl_fcnn.pth')
fcnn_preds, fcnn_probs = get_predictions(fcnn_model, 'fcnn')
lf_probs = best_w * all_probs['LateFusion_TL'] + (1 - best_w) * fcnn_probs
all_preds['LateFusion_TL'] = lf_probs.argmax(1)
all_probs['LateFusion_TL'] = lf_probs
f1_lf = f1_score(y_te, all_preds['LateFusion_TL'], average='macro', zero_division=0)
print(f'LateFusion_TL (ensemble): macro F1 = {f1_lf:.4f}')

In [ ]:
# ── Identify misclassified samples per model per class ──────────────────────
N_SAMPLES = 3  # jumlah sampel per kelas untuk ditampilkan
rng = np.random.RandomState(42)

def sample_indices(preds, true_cls, pred_cls, n=N_SAMPLES):
    """Ambil n sampel yang true_cls diprediksi sebagai pred_cls."""
    mask = (y_te == true_cls) & (preds == pred_cls)
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return []
    return rng.choice(idx, size=min(n, len(idx)), replace=False).tolist()

# Print misclassification summary per model
for name in models:
    preds = all_preds[name]
    correct = (preds == y_te).sum()
    wrong   = (preds != y_te).sum()
    print(f'\n{name}: correct={correct}  wrong={wrong}')
    cm = confusion_matrix(y_te, preds)
    print('Confusion matrix (true \ pred):')
    print(f'  {"":12s}  {" ".join(f"{c:>10s}" for c in CLASS_NAMES)}')
    for i, row in enumerate(cm):
        print(f'  {CLASS_NAMES[i]:12s}  {" ".join(f"{v:>10d}" for v in row)}')

In [ ]:
# ── Grad-CAM helper ──────────────────────────────────────────────────────────
def get_target_layer(model, model_name):
    """Ambil layer4 last block dari ResNet18 backbone."""
    if model_name in ('CNN_TL', 'LateFusion_TL'):
        return model.features[-2][-1]  # ResNet18 layer4 last residual block
    elif model_name == 'EarlyFusion_TL':
        return model.features[-2][-1]
    elif model_name == 'Intermediate_TL':
        return model.image_features[-2][-1]  # CNN branch in intermediate fusion
    raise ValueError(f'Unknown model: {model_name}')


def run_gradcam_single(model, model_name, model_type, idx):
    """
    Jalankan Grad-CAM untuk satu sampel.
    Returns: (rgb_image H×W×3 float32, cam_overlay H×W×3 float32)
    """
    target_layer = get_target_layer(model, model_name)
    cam = GradCAM(model=model, target_layers=[target_layer])

    # Prepare input
    if model_type == 'cnn':
        input_tensor = img_tensor([idx])  # (1,3,H,W)
    elif model_type == 'early_fusion':
        input_tensor = ef_tensor([idx])   # (1,4,H,W)
    elif model_type == 'fusion':
        # IntermediateFusion: wrap so GradCAM only sees image input
        import functools
        lm_fixed = lm_tensor([idx])  # fix landmark
        original_forward = model.forward
        model.forward = functools.partial(original_forward.__func__, model,
                                          landmark=lm_fixed) if False else model.forward
        # Simpler: create wrapper
        class FusionWrapper(torch.nn.Module):
            def __init__(self, m, lm):
                super().__init__()
                self.m = m
                self.lm = lm
            def forward(self, x):
                return self.m(x, self.lm)
        lm_fixed = lm_tensor([idx])
        wrapper = FusionWrapper(model, lm_fixed)
        target_layer = model.image_features[-2][-1]
        cam = GradCAM(model=wrapper, target_layers=[target_layer])
        input_tensor = img_tensor([idx])

    # Normalize image for overlay
    rgb = img_te[idx].copy()  # H×W×3 float32
    # Normalize to [0,1] if needed
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    rgb = np.clip(rgb, 0, 1)

    grayscale_cam = cam(input_tensor=input_tensor, targets=None)  # use predicted class
    overlay = show_cam_on_image(rgb, grayscale_cam[0], use_rgb=True)
    return rgb, overlay


print('Grad-CAM helpers defined.')

In [ ]:
# ── Plot 1: Grad-CAM per model — benar vs salah per kelas ───────────────────
# Untuk setiap model: pilih 1 sampel benar + 1 salah untuk setiap true class

for model_name, model in models.items():
    model_type = model_types[model_name]
    preds = all_preds[model_name]

    fig, axes = plt.subplots(NUM_CLASSES * 2, 4, figsize=(14, NUM_CLASSES * 4))
    fig.suptitle(f'{model_name} — Grad-CAM Error Analysis (3-class Primer)',
                 fontsize=13, fontweight='bold', y=1.01)

    col_labels = ['Original', 'Grad-CAM (benar)', 'Original', 'Grad-CAM (salah)']
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=9, fontweight='bold')

    for cls in range(NUM_CLASSES):
        row_base = cls * 2

        # Correct: predicted == true
        correct_idx = sample_indices(preds, cls, cls, n=1)
        # Wrong: predicted != true (take most common error)
        wrong_idx = []
        for pred_cls in range(NUM_CLASSES):
            if pred_cls == cls: continue
            w = sample_indices(preds, cls, pred_cls, n=1)
            if w:
                wrong_idx = w
                wrong_pred = pred_cls
                break

        # Row 1: correct
        ax = axes[row_base]
        ax[0].set_ylabel(f'True: {CLASS_NAMES[cls]}\n(benar)', fontsize=8)
        if correct_idx:
            i = correct_idx[0]
            rgb, overlay = run_gradcam_single(model, model_name, model_type, i)
            conf = all_probs[model_name][i][cls]
            ax[0].imshow(rgb); ax[0].axis('off')
            ax[0].set_title(f'conf={conf:.2f}', fontsize=7)
            ax[1].imshow(overlay); ax[1].axis('off')
            ax[1].set_title(f'→ pred: {CLASS_NAMES[cls]} ✓', fontsize=7, color='green')
        else:
            ax[0].text(0.5, 0.5, 'No sample', ha='center', va='center')
            ax[0].axis('off'); ax[1].axis('off')

        # Row 2: wrong
        ax2 = axes[row_base + 1] if row_base + 1 < len(axes) else None
        if ax2 is not None:
            ax2[0].set_ylabel(f'True: {CLASS_NAMES[cls]}\n(salah)', fontsize=8)
            if wrong_idx:
                i = wrong_idx[0]
                rgb, overlay = run_gradcam_single(model, model_name, model_type, i)
                conf = all_probs[model_name][i][wrong_pred]
                ax2[0].imshow(rgb); ax2[0].axis('off')
                ax2[0].set_title(f'conf={conf:.2f}', fontsize=7)
                ax2[1].imshow(overlay); ax2[1].axis('off')
                ax2[1].set_title(f'→ pred: {CLASS_NAMES[wrong_pred]} ✗', fontsize=7, color='red')
                ax2[2].axis('off'); ax2[3].axis('off')
            else:
                for a in ax2: a.axis('off')

    plt.tight_layout()
    save_path = OUT_DIR / f'gradcam_{model_name.lower()}_error.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
# ── Plot 2: Perbandingan 4 model untuk sampel yang SAMA ─────────────────────
# Pilih sampel yang SEMUA model salah prediksi → paling informatif

# Sampel yang salah di semua model
wrong_all = np.ones(len(y_te), dtype=bool)
for name in models:
    wrong_all &= (all_preds[name] != y_te)

print(f'Sampel salah di SEMUA 4 model: {wrong_all.sum()}')

for cls in range(NUM_CLASSES):
    cls_mask = (y_te == cls) & wrong_all
    idxs = np.where(cls_mask)[0]
    if len(idxs) == 0:
        print(f'  class {CLASS_NAMES[cls]}: tidak ada sampel salah di semua model')
        continue
    sel = rng.choice(idxs, size=min(3, len(idxs)), replace=False)

    for sample_idx in sel:
        fig, axes = plt.subplots(1, 5, figsize=(18, 3))
        fig.suptitle(
            f'True: {CLASS_NAMES[cls]} | Sample #{sample_idx} — semua model salah',
            fontsize=11, fontweight='bold'
        )

        # Col 0: original
        rgb = np.clip(img_te[sample_idx].copy(), 0, 1)
        if rgb.max() > 1.0: rgb = rgb / 255.0
        axes[0].imshow(rgb); axes[0].axis('off'); axes[0].set_title('Original', fontsize=9)

        # Cols 1-4: Grad-CAM per model
        for col, (model_name, model) in enumerate(models.items(), start=1):
            model_type = model_types[model_name]
            pred_cls = all_preds[model_name][sample_idx]
            conf = all_probs[model_name][sample_idx][pred_cls]
            _, overlay = run_gradcam_single(model, model_name, model_type, sample_idx)
            axes[col].imshow(overlay)
            axes[col].axis('off')
            axes[col].set_title(
                f'{model_name}\n→ {CLASS_NAMES[pred_cls]} ({conf:.2f})',
                fontsize=8, color='red'
            )

        plt.tight_layout()
        save_path = OUT_DIR / f'gradcam_comparison_cls{cls}_sample{sample_idx}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {save_path}')

In [ ]:
# ── Ringkasan metrics semua model ────────────────────────────────────────────
from sklearn.metrics import classification_report

print('='*60)
print('RINGKASAN METRICS — 3-class Primer test set')
print('='*60)
for name in models:
    preds = all_preds[name]
    f1_macro = f1_score(y_te, preds, average='macro', zero_division=0)
    f1_w     = f1_score(y_te, preds, average='weighted', zero_division=0)
    print(f'\n{name}  macro_f1={f1_macro:.4f}  weighted_f1={f1_w:.4f}')
    print(classification_report(y_te, preds, target_names=CLASS_NAMES, zero_division=0))